# 화이트 벨런스 체크

In [ ]:
import cv2
import numpy as np
import os
import random
import matplotlib.pyplot as plt

# ==========================================
# 📂 경로 설정
# ==========================================
img_root = r"C:\Users\home\Desktop\AI_study\data\train_images"

def apply_white_balance(img, alpha=0.3):
    """
    Gray World 가설 기반 화이트 밸런스 + Alpha Blending
    alpha=0 이면 원본, alpha=1 이면 100% 보정본
    """
    # 1. 보정본 생성 (Gray World)
    result = img.astype(np.float32)
    avg_b = np.mean(result[:, :, 0])
    avg_g = np.mean(result[:, :, 1])
    avg_r = np.mean(result[:, :, 2])
    avg_gray = (avg_b + avg_g + avg_r) / 3

    result[:, :, 0] *= (avg_gray / avg_b)
    result[:, :, 1] *= (avg_gray / avg_g)
    result[:, :, 2] *= (avg_gray / avg_r)
    result = np.clip(result, 0, 255).astype(np.uint8)

    # 2. 원본과 보정본 합성 (Alpha Blending)
    final = cv2.addWeighted(img, 1 - alpha, result, alpha, 0)
    return final

def run_comparison_test(image_name=None):
    # 이미지 선택
    if image_name:
        # 특정 이미지를 보고 싶을 때 (예: 노란 알약 파일명)
        img_path = None
        for root, dirs, files in os.walk(img_root):
            if image_name in files:
                img_path = os.path.join(root, image_name)
                break
    else:
        # 무작위 이미지 선택
        all_imgs = []
        for root, dirs, files in os.walk(img_root):
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    all_imgs.append(os.path.join(root, f))
        img_path = random.choice(all_imgs)

    if not img_path:
        print("❌ 이미지를 찾을 수 없습니다.")
        return

    # 이미지 읽기
    original = cv2.imread(img_path)
    if original is None: return
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB) # 출력을 위해 RGB 변환

    # 비교 리스트 생성 (Alpha: 0.0, 0.1, 0.2, 0.3, 0.5)
    alphas = [0.0, 0.1, 0.2, 0.3, 0.5]
    results = []

    for a in alphas:
        res = apply_white_balance(original, alpha=a)
        # 텍스트 삽입 (이미지에 강도 표시)
        cv2.putText(res, f"Alpha: {a}", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 
                    4, (255, 255, 255), 10, cv2.LINE_AA)
        results.append(res)

    # 시각화 (이미지가 너무 크면 리사이즈해서 출력)
    combined = np.hstack(results) # 가로로 이어붙이기
    plt.figure(figsize=(25, 10))
    plt.imshow(combined)
    plt.axis('off')
    plt.title(f"White Balance Strength Test (File: {os.path.basename(img_path)})")
    plt.tight_layout()
    plt.show()

# 실행: 특정 파일명을 넣거나(예: 'K-012345_0_0_0_0_0.jpg'), 빈칸으로 두어 무작위로 확인하세요.
run_comparison_test()

큰 차이는 없어 보였습니다. 

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

def apply_preprocessing(img_path, alpha=0.5):
    # 1. 원본 이미지 로드
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 2. CLAHE (대비 최적화)
    # 각인을 도드라지게 함
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    clahe_img = cv2.merge((l, a, b))
    clahe_img = cv2.cvtColor(clahe_img, cv2.COLOR_LAB2RGB)

    # 3. White Balance (Gray World + Alpha 조절)
    # 왜곡을 막기 위해 원본과 alpha 비율로 합성
    result_wb = img.astype(float)
    avg_b = np.average(result_wb[:, :, 0])
    avg_g = np.average(result_wb[:, :, 1])
    avg_r = np.average(result_wb[:, :, 2])
    avg_gray = (avg_b + avg_g + avg_r) / 3

    result_wb[:, :, 0] *= (avg_gray / avg_b)
    result_wb[:, :, 1] *= (avg_gray / avg_g)
    result_wb[:, :, 2] *= (avg_gray / avg_r)
    result_wb = np.clip(result_wb, 0, 255).astype(np.uint8)
    
    # 원본과 합성 (왜곡 방지)
    wb_final = cv2.addWeighted(img, 1 - alpha, result_wb, alpha, 0)
    wb_final = cv2.cvtColor(wb_final, cv2.COLOR_BGR2RGB)

    # 4. Sharpening (선명화)
    # 경계선을 뚜렷하게 함
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    sharp_img = cv2.filter2D(img_rgb, -1, kernel)

    return img_rgb, clahe_img, wb_final, sharp_img

# --- 실행부 ---
# 사용자님의 크롭 이미지 폴더 경로를 넣어주세요
sample_path = r"C:\Users\home\Desktop\AI_study\processed_crops_old_ONLY1.2"
image_files = glob.glob(os.path.join(sample_path, "**", "*.jpg"), recursive=True)[:3] # 샘플 3개만

plt.figure(figsize=(20, 12))

for i, f_path in enumerate(image_files):
    results = apply_preprocessing(f_path, alpha=0.6) # alpha 0.6: 화이트밸런스 60% 적용
    titles = ['Original', 'CLAHE', 'White Balance (Gray World)', 'Sharpening']
    
    for j, res in enumerate(results):
        plt.subplot(len(image_files), 4, i * 4 + j + 1)
        plt.imshow(res)
        plt.title(f"{titles[j]} - Sample {i+1}")
        plt.axis('off')

plt.tight_layout()
plt.show()

여기서도, CLAHE, Sharpening은 효과적인지만, 화이트 밸런스는 영향이 없어보입니다. 

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import random

# ==========================================
# 🔧 전처리 함수 정의 (강도 조절 가능)
# ==========================================

def apply_clahe(img_lab, clip_limit):
    """CLAHE 적용 (clip_limit: 대비 제한 강도, 높을수록 강함)"""
    l, a, b = cv2.split(img_lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    return cv2.merge((l_clahe, a, b))

def apply_gray_world(img_bgr, alpha):
    """Gray World + 원본 합성 (alpha: 0~1, 1에 가까울수록 GW 효과 강함)"""
    img_float = img_bgr.astype(float)
    avg_b = np.average(img_float[:, :, 0])
    avg_g = np.average(img_float[:, :, 1])
    avg_r = np.average(img_float[:, :, 2])
    
    # 0으로 나누는 오류 방지
    if avg_b == 0 or avg_g == 0 or avg_r == 0:
        return img_bgr

    avg_gray = (avg_b + avg_g + avg_r) / 3
    img_float[:, :, 0] *= (avg_gray / avg_b)
    img_float[:, :, 1] *= (avg_gray / avg_g)
    img_float[:, :, 2] *= (avg_gray / avg_r)
    gw_result = np.clip(img_float, 0, 255).astype(np.uint8)
    
    # 원본과 합성하여 왜곡 완화
    return cv2.addWeighted(img_bgr, 1 - alpha, gw_result, alpha, 0)

def apply_sharpening(img_rgb, strength_level):
    """Sharpening 적용 (strength_level: 1=약, 2=중, 3=강)"""
    # 강도별 커널 정의 (중앙값이 클수록, 주변과의 차이가 클수록 강해짐)
    if strength_level == 1: # 약
        kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    elif strength_level == 2: # 중
        kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])
    else: # 강 (매우 강함)
        kernel = np.array([[-1, -2, -1], [-2, 13, -2], [-1, -2, -1]])
        
    return cv2.filter2D(img_rgb, -1, kernel)

# ==========================================
# 📂 경로 설정 및 샘플 추출 (사용자 수정 영역)
# ==========================================
# 크롭 이미지가 있는 최상위 폴더 경로
crop_root = r"C:\Users\home\Desktop\AI_study\processed_crops_old_ONLY1.2"

# 약물 ID별로 폴더를 찾고, 각 폴더에서 첫 번째 이미지를 샘플로 선택
drug_folders = [f for f in glob.glob(os.path.join(crop_root, "*")) if os.path.isdir(f)]
random.shuffle(drug_folders) # 랜덤으로 섞어서 다양한 약 선택
selected_folders = drug_folders[:5] # 5개 약물 선택

sample_images = []
drug_ids = []
for folder in selected_folders:
    imgs = glob.glob(os.path.join(folder, "*.jpg"))
    if imgs:
        sample_images.append(imgs[0]) # 첫 번째 이미지 선택
        drug_ids.append(os.path.basename(folder))

print(f"🧪 선택된 샘플 약물 ID: {drug_ids}")

# ==========================================
# 🖼️ 시각화 실행
# ==========================================
plt.figure(figsize=(25, 15)) # 전체 그림 크기 설정 (가로로 길게)

rows = len(sample_images)
cols = 10 # 원본(1) + CLAHE(3) + GW(3) + Sharp(3)

# 강도 설정값 (테스트용)
clahe_limits = [1.0, 3.0, 6.0] # 약, 중, 강
gw_alphas = [0.3, 0.6, 1.0]    # 30%, 60%, 100%(완전 적용)
sharp_levels = [1, 2, 3]       # 약, 중, 강 커널

for i, img_path in enumerate(sample_images):
    # 1. 이미지 로드 및 색상 공간 변환
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

    processed_imgs = []
    titles = []

    # --- (1) 원본 ---
    processed_imgs.append(img_rgb)
    titles.append("Original")

    # --- (2) CLAHE 3단계 ---
    for limit in clahe_limits:
        res_lab = apply_clahe(img_lab.copy(), limit)
        processed_imgs.append(cv2.cvtColor(res_lab, cv2.COLOR_LAB2RGB))
        titles.append(f"CLAHE (Limit={limit})")

    # --- (3) Gray World 3단계 ---
    for alpha in gw_alphas:
        res_bgr = apply_gray_world(img_bgr.copy(), alpha)
        processed_imgs.append(cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB))
        titles.append(f"Gray World (Alpha={alpha})")

    # --- (4) Sharpening 3단계 ---
    for level in sharp_levels:
        processed_imgs.append(apply_sharpening(img_rgb.copy(), level))
        titles.append(f"Sharpening (Lv.{level})")

    # --- 한 줄에 출력 ---
    for j, (proc_img, title) in enumerate(zip(processed_imgs, titles)):
        idx = i * cols + j + 1
        plt.subplot(rows, cols, idx)
        plt.imshow(proc_img)
        if i == 0: # 첫 번째 줄에만 타이틀 표시
            plt.title(title, fontsize=10, fontweight='bold')
        if j == 0: # 첫 번째 열에만 약물 ID 표시
            plt.ylabel(drug_ids[i], fontsize=10, fontweight='bold')
        plt.xticks([]), plt.yticks([]) # 축 눈금 제거

plt.tight_layout()
plt.subplots_adjust(wspace=0.1, hspace=0.1) # 간격 조절
plt.show()

### Gray World는 크게 영향 없을 것 같고, CLAHE와 선명화 효과만 적용하기로 결정

# [CLAHE(3, 6) + Bilateral(약, 강) + Sharpening(Lv3 고정)] 조합에 대한 10개의 랜덤 샘플

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import random

# ==========================================
# 🔧 전처리 함수 정의
# ==========================================

def apply_pipeline(img_bgr, clahe_limit, bi_level):
    """
    파이프라인 순서: CLAHE (대비) -> Bilateral (노이즈 제거) -> Sharpening (선명화)
    """
    # 1. CLAHE 적용 (L 채널에만 적용하여 색상 왜곡 방지)
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(img_lab)
    clahe = cv2.createCLAHE(clipLimit=clahe_limit, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_clahe = cv2.merge((l, a, b))
    img_clahe = cv2.cvtColor(img_clahe, cv2.COLOR_LAB2BGR) # 다시 BGR로

    # 2. Bilateral Filter 적용 (노이즈 제거)
    # d: 필터링에 이용하는 이웃 픽셀의 지름 (높을수록 많이 뭉갬)
    # sigmaColor: 색공간에서 얼마나 차이나는 것까지 섞을지 (높을수록 뭉개짐)
    # sigmaSpace: 좌표공간에서 얼마나 멀리 있는 것까지 섞을지
    if bi_level == 1: # 약한 노이즈 제거
        img_bi = cv2.bilateralFilter(img_clahe, d=5, sigmaColor=50, sigmaSpace=50)
    else: # 강한 노이즈 제거
        img_bi = cv2.bilateralFilter(img_clahe, d=9, sigmaColor=100, sigmaSpace=100)

    # 3. Sharpening 적용 (Lv3 강도 고정)
    # 노이즈를 먼저 잡았으므로 강하게 때려도 노이즈가 덜 부각됨
    kernel_strong = np.array([[-1, -2, -1], [-2, 13, -2], [-1, -2, -1]])
    img_result = cv2.filter2D(img_bi, -1, kernel_strong)
    
    return cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB)

# ==========================================
# 📂 데이터 로드 및 랜덤 샘플링
# ==========================================
crop_root = r"C:\Users\home\Desktop\AI_study\processed_crops_old_ONLY1.2"

# 약물 폴더 리스트업
all_drug_folders = [f for f in glob.glob(os.path.join(crop_root, "*")) if os.path.isdir(f)]

# 랜덤으로 10개 선택 (폴더가 10개 미만이면 전체 선택)
num_samples = 10
if len(all_drug_folders) >= num_samples:
    selected_folders = random.sample(all_drug_folders, num_samples)
else:
    selected_folders = all_drug_folders

sample_images = []
drug_ids = []

for folder in selected_folders:
    imgs = glob.glob(os.path.join(folder, "*.jpg"))
    if imgs:
        # 각 폴더 내에서도 랜덤으로 1장 선택
        sample_images.append(random.choice(imgs))
        drug_ids.append(os.path.basename(folder))

print(f"🎲 랜덤 선택된 약물 {len(drug_ids)}종: {drug_ids}")

# ==========================================
# 🖼️ 시각화 실행
# ==========================================
# 행: 약물 종류(10개), 열: 원본 + 4가지 조합
plt.figure(figsize=(20, 40)) 

cols = 5
col_titles = [
    "Original",
    "CLAHE(3.0) + Bi(Low)",   # 적당한 대비 + 약한 노이즈제거
    "CLAHE(3.0) + Bi(High)",  # 적당한 대비 + 강한 노이즈제거
    "CLAHE(6.0) + Bi(Low)",   # 강한 대비 + 약한 노이즈제거 (거칠 수 있음)
    "CLAHE(6.0) + Bi(High)"   # 강한 대비 + 강한 노이즈제거 (가장 드라마틱)
]

for i, img_path in enumerate(sample_images):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # 4가지 조합 생성
    # 파라미터: (이미지, CLAHE강도, Bilateral강도)
    res1 = apply_pipeline(img_bgr, clahe_limit=3.0, bi_level=1)
    res2 = apply_pipeline(img_bgr, clahe_limit=3.0, bi_level=2)
    res3 = apply_pipeline(img_bgr, clahe_limit=6.0, bi_level=1)
    res4 = apply_pipeline(img_bgr, clahe_limit=6.0, bi_level=2)
    
    display_list = [img_rgb, res1, res2, res3, res4]
    
    for j, img in enumerate(display_list):
        idx = i * cols + j + 1
        plt.subplot(len(sample_images), cols, idx)
        plt.imshow(img)
        
        # 첫 번째 행에만 타이틀 달기
        if i == 0:
            plt.title(col_titles[j], fontsize=12, fontweight='bold')
        
        # 첫 번째 열에 약물 ID 표시
        if j == 0:
            plt.ylabel(drug_ids[i], fontsize=11, fontweight='bold')
            
        plt.xticks([]), plt.yticks([])

plt.tight_layout()
plt.show()

# [CLAHE(1.5, 2.0, 2.5, 3.0) + Bilateral(1.0) ] 조합을 10개의 랜덤 샘플

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import random

# ==========================================
# 🔧 전처리 함수 정의
# ==========================================

def apply_pipeline(img_bgr, clahe_limit, bi_level):
    """
    파이프라인 순서: CLAHE (대비) -> Bilateral (노이즈 제거) -> Sharpening (선명화)
    """
    # 1. CLAHE 적용 (L 채널에만 적용하여 색상 왜곡 방지)
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(img_lab)
    clahe = cv2.createCLAHE(clipLimit=clahe_limit, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_clahe = cv2.merge((l, a, b))
    img_clahe = cv2.cvtColor(img_clahe, cv2.COLOR_LAB2BGR) # 다시 BGR로

    # 2. Bilateral Filter 적용 (노이즈 제거)
    # d: 필터링에 이용하는 이웃 픽셀의 지름 (높을수록 많이 뭉갬)
    # sigmaColor: 색공간에서 얼마나 차이나는 것까지 섞을지 (높을수록 뭉개짐)
    # sigmaSpace: 좌표공간에서 얼마나 멀리 있는 것까지 섞을지
    if bi_level == 1: # 약한 노이즈 제거
        img_bi = cv2.bilateralFilter(img_clahe, d=5, sigmaColor=50, sigmaSpace=50)
    else: # 강한 노이즈 제거
        img_bi = cv2.bilateralFilter(img_clahe, d=9, sigmaColor=100, sigmaSpace=100)

    # 3. Sharpening 적용 (Lv3 강도 고정)
    # 노이즈를 먼저 잡았으므로 강하게 때려도 노이즈가 덜 부각됨
    kernel_strong = np.array([[-1, -2, -1], [-2, 13, -2], [-1, -2, -1]])
    img_result = cv2.filter2D(img_bi, -1, kernel_strong)
    
    return cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB)

# ==========================================
# 📂 데이터 로드 및 랜덤 샘플링
# ==========================================
crop_root = r"C:\Users\home\Desktop\AI_study\processed_crops_old_ONLY1.2"

# 약물 폴더 리스트업
all_drug_folders = [f for f in glob.glob(os.path.join(crop_root, "*")) if os.path.isdir(f)]

# 랜덤으로 10개 선택 (폴더가 10개 미만이면 전체 선택)
num_samples = 10
if len(all_drug_folders) >= num_samples:
    selected_folders = random.sample(all_drug_folders, num_samples)
else:
    selected_folders = all_drug_folders

sample_images = []
drug_ids = []

for folder in selected_folders:
    imgs = glob.glob(os.path.join(folder, "*.jpg"))
    if imgs:
        # 각 폴더 내에서도 랜덤으로 1장 선택
        sample_images.append(random.choice(imgs))
        drug_ids.append(os.path.basename(folder))

print(f"🎲 랜덤 선택된 약물 {len(drug_ids)}종: {drug_ids}")

# ==========================================
# 🖼️ 시각화 실행
# ==========================================
# 행: 약물 종류(10개), 열: 원본 + 4가지 조합
plt.figure(figsize=(20, 40)) 

cols = 5
col_titles = [
    "Original",
    "CLAHE(1.5) + Bi(Low)",   # 적당한 대비 + 약한 노이즈제거
    "CLAHE(2.0) + Bi(Low)",  # 적당한 대비 + 강한 노이즈제거
    "CLAHE(2.5) + Bi(Low)",   # 강한 대비 + 약한 노이즈제거 (거칠 수 있음)
    "CLAHE(3.0) + Bi(Low)"   # 강한 대비 + 강한 노이즈제거 (가장 드라마틱)
]

for i, img_path in enumerate(sample_images):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # 4가지 조합 생성
    # 파라미터: (이미지, CLAHE강도, Bilateral강도)
    res1 = apply_pipeline(img_bgr, clahe_limit=1.5, bi_level=1)
    res2 = apply_pipeline(img_bgr, clahe_limit=2.0, bi_level=1)
    res3 = apply_pipeline(img_bgr, clahe_limit=2.5, bi_level=1)
    res4 = apply_pipeline(img_bgr, clahe_limit=3.0, bi_level=1)
    
    display_list = [img_rgb, res1, res2, res3, res4]
    
    for j, img in enumerate(display_list):
        idx = i * cols + j + 1
        plt.subplot(len(sample_images), cols, idx)
        plt.imshow(img)
        
        # 첫 번째 행에만 타이틀 달기
        if i == 0:
            plt.title(col_titles[j], fontsize=12, fontweight='bold')
        
        # 첫 번째 열에 약물 ID 표시
        if j == 0:
            plt.ylabel(drug_ids[i], fontsize=11, fontweight='bold')
            
        plt.xticks([]), plt.yticks([])

plt.tight_layout()
plt.show()

CLAHE 2.0이 제일 좋습니다. 다만, 배경 노이즈 제거가 필요해보입니다. 
이를 위해 Bilateral Filter를 강화하지만, 글자와 테두리가 뭉게지는 것을 방지하기 위해, 선명화를 적절히 활용할 예정입니다. 


다음과 같은 체크리스트를 활용하여 육안 검사를 진행합니다.

🔍 알약 각인 육안 검수 체크리스트
[ ] 획의 연속성: 글자 획이 중간에 끊기지 않고 선으로 이어져 있는가?

[ ] 구멍의 개방: '0', '8' 등의 내부 구멍이 뭉개지지 않고 잘 뚫려 있는가?

[ ] 링잉 현상: 글자 테두리에 하얀 빛(헤일로)이 과하게 생기지 않았는가?

[ ] 주인공 대비: 실눈 뜨고 볼 때 배경 노이즈보다 글자가 더 선명한가?

[ ] 경계선 분리: 알약 가장자리와 바닥 배경이 뚜렷하게 구분되는가?

[ ] 디테일 보존: 각인의 미세한 굴곡이나 음각 깊이가 느껴지는가?

[ ] 질감 과도기: 표면이 너무 뭉개져서 플라스틱/만화처럼 보이지 않는가?

[ ] 색상 무결성: 흰색 약이 보라색이나 녹색으로 변색되지 않았는가?



# [CLAHE 2.0 고정] + [Bi 1~2] + [Sharp 3~5] 랜덤 조합 테스트

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import random

# ==========================================
# 🔧 전처리 함수 정의 (사용자 요청 사항 반영)
# ==========================================

def get_processed_image(img_bgr, bi_level, sharp_level):
    # 1. CLAHE 적용 (고정값 2.0)
    # 색상 왜곡 방지를 위해 L 채널에만 적용
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(img_lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_clahe = cv2.merge((l, a, b))
    img_clahe = cv2.cvtColor(img_clahe, cv2.COLOR_LAB2BGR)

    # 2. Bilateral Filter (노이즈 제거)
    if bi_level == 1: # 약하게 뭉개기
        img_bi = cv2.bilateralFilter(img_clahe, d=5, sigmaColor=50, sigmaSpace=50)
    else: # 강하게 뭉개기 (Level 2)
        img_bi = cv2.bilateralFilter(img_clahe, d=9, sigmaColor=100, sigmaSpace=100)

    # 3. Sharpening (선명화 - 강도 3, 4, 5)
    if sharp_level == 3: # 기존 강함
        kernel = np.array([[-1, -2, -1], 
                           [-2, 13, -2], 
                           [-1, -2, -1]])
    elif sharp_level == 4: # 매우 강함 (더 날카로움)
        kernel = np.array([[-1, -3, -1], 
                           [-3, 17, -3], 
                           [-1, -3, -1]])
    elif sharp_level == 5: # 극한 (노이즈 감수하고 엣지 극대화)
        kernel = np.array([[-2, -2, -2], 
                           [-2, 17, -2], 
                           [-2, -2, -2]]) # 중앙값 강조
    else:
        return cv2.cvtColor(img_bi, cv2.COLOR_BGR2RGB)

    # 필터 적용
    img_result = cv2.filter2D(img_bi, -1, kernel)
    return cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB)

# ==========================================
# 📂 데이터 로드 및 랜덤 샘플링 (10개)
# ==========================================
crop_root = r"C:\Users\home\Desktop\AI_study\processed_crops_old_ONLY1.2"

# 전체 약품 폴더 찾기
all_drug_folders = [f for f in glob.glob(os.path.join(crop_root, "*")) if os.path.isdir(f)]

# 랜덤 10개 선택
num_samples = 10
if len(all_drug_folders) >= num_samples:
    selected_folders = random.sample(all_drug_folders, num_samples)
else:
    selected_folders = all_drug_folders

sample_images = []
drug_ids = []

for folder in selected_folders:
    imgs = glob.glob(os.path.join(folder, "*.jpg"))
    if imgs:
        sample_images.append(random.choice(imgs)) # 폴더 내 랜덤 1장
        drug_ids.append(os.path.basename(folder))

print(f"🧪 테스트 약물 {len(drug_ids)}종: {drug_ids}")

# ==========================================
# 🖼️ 시각화 실행 (10행 7열)
# ==========================================
plt.figure(figsize=(24, 40)) # 세로로 긴 이미지

# 컬럼 타이틀 (조합 설명)
col_titles = [
    "Original",
    "Bi(1) + Sharp(3)",
    "Bi(1) + Sharp(4)",
    "Bi(1) + Sharp(5)",
    "Bi(2) + Sharp(3)",
    "Bi(2) + Sharp(4)",
    "Bi(2) + Sharp(5)"
]

# 조합 리스트 [(Bi, Sharp)]
combinations = [
    (1, 3), (1, 4), (1, 5),
    (2, 3), (2, 4), (2, 5)
]

for i, img_path in enumerate(sample_images):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # 1. 원본
    images_to_show = [img_rgb]
    
    # 2. 6가지 조합 생성
    for bi, sharp in combinations:
        processed = get_processed_image(img_bgr, bi_level=bi, sharp_level=sharp)
        images_to_show.append(processed)
    
    # 출력
    for j, img in enumerate(images_to_show):
        idx = i * 7 + j + 1
        plt.subplot(len(sample_images), 7, idx)
        plt.imshow(img)
        
        # 첫 번째 행에만 타이틀 표시
        if i == 0:
            plt.title(col_titles[j], fontsize=14, fontweight='bold', pad=15)
        
        # 첫 번째 열에 약물 ID 표시
        if j == 0:
            plt.ylabel(drug_ids[i], fontsize=12, fontweight='bold', rotation=90)
            
        plt.xticks([]), plt.yticks([])

plt.tight_layout()
plt.subplots_adjust(wspace=0.1, hspace=0.1) # 간격 미세 조정
plt.show()

BI는 1이 훨씬 낫다고 판단, Sharpening은 높을수록 좋아서, 수치를 더 조정할 예정.
CLAHE 2.0, Bilateral Filter = 1로 고정 

# 선명화 효과(Sharpening) 확인을 위한 10개 랜덤 조합 

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import random

# ==========================================
# 🔧 전처리 함수 (Sharpening 강도 세분화)
# ==========================================

def get_processed_image(img_bgr, sharp_level):
    # 1. CLAHE 적용 (고정값 2.0)
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(img_lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_clahe = cv2.merge((l, a, b))
    img_clahe = cv2.cvtColor(img_clahe, cv2.COLOR_LAB2BGR)

    # 2. Bilateral Filter (고정값 1: 약하게 뭉개기)
    # 노이즈를 살짝만 잡아주고 엣지를 최대한 보존
    img_bi = cv2.bilateralFilter(img_clahe, d=5, sigmaColor=50, sigmaSpace=50)

    # 3. Sharpening (강도 3 ~ 8)
    if sharp_level == 3: # 강함 (기본 High)
        kernel = np.array([[-1, -1, -1], 
                           [-1, 9, -1], 
                           [-1, -1, -1]])
    elif sharp_level == 4: # 매우 강함
        kernel = np.array([[-1, -2, -1], 
                           [-2, 13, -2], 
                           [-1, -2, -1]])
    elif sharp_level == 5: # 극한
        kernel = np.array([[-2, -2, -2], 
                           [-2, 17, -2], 
                           [-2, -2, -2]])
    elif sharp_level == 6: # 초극한 (여기서부터 노이즈 폭발 가능성)
        kernel = np.array([[-2, -3, -2], 
                           [-3, 21, -3], 
                           [-2, -3, -2]])
    elif sharp_level == 7: # 붕괴 직전
        kernel = np.array([[-3, -3, -3], 
                           [-3, 25, -3], 
                           [-3, -3, -3]])
    elif sharp_level == 8: # 핵폭탄 (Nuclear)
        kernel = np.array([[-4, -4, -4], 
                           [-4, 33, -4], 
                           [-4, -4, -4]])
    else:
        return cv2.cvtColor(img_bi, cv2.COLOR_BGR2RGB)

    # 필터 적용
    img_result = cv2.filter2D(img_bi, -1, kernel)
    return cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB)

# ==========================================
# 📂 데이터 로드 (랜덤 10개)
# ==========================================
crop_root = r"C:\Users\home\Desktop\AI_study\processed_crops_old_ONLY1.2"

all_drug_folders = [f for f in glob.glob(os.path.join(crop_root, "*")) if os.path.isdir(f)]

num_samples = 10
if len(all_drug_folders) >= num_samples:
    selected_folders = random.sample(all_drug_folders, num_samples)
else:
    selected_folders = all_drug_folders

sample_images = []
drug_ids = []

for folder in selected_folders:
    imgs = glob.glob(os.path.join(folder, "*.jpg"))
    if imgs:
        sample_images.append(random.choice(imgs))
        drug_ids.append(os.path.basename(folder))

print(f"🧪 테스트 약물 {len(drug_ids)}종: {drug_ids}")

# ==========================================
# 🖼️ 시각화 실행 (10행 7열)
# ==========================================
plt.figure(figsize=(24, 40))

col_titles = [
    "Original",
    "Sharp(3) - Strong",
    "Sharp(4) - Very Strong",
    "Sharp(5) - Extreme",
    "Sharp(6) - Ultra",
    "Sharp(7) - Mega",
    "Sharp(8) - Nuclear"
]

sharp_levels = [3, 4, 5, 6, 7, 8]

for i, img_path in enumerate(sample_images):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # 리스트에 원본 먼저 담기
    images_to_show = [img_rgb]
    
    # 단계별 처리 이미지 담기
    for level in sharp_levels:
        processed = get_processed_image(img_bgr, sharp_level=level)
        images_to_show.append(processed)
    
    # 출력
    for j, img in enumerate(images_to_show):
        idx = i * 7 + j + 1
        plt.subplot(len(sample_images), 7, idx)
        plt.imshow(img)
        
        # 타이틀 (첫 줄만)
        if i == 0:
            plt.title(col_titles[j], fontsize=13, fontweight='bold', pad=15)
        
        # 약물 ID (첫 열만)
        if j == 0:
            plt.ylabel(drug_ids[i], fontsize=12, fontweight='bold', rotation=90)
            
        plt.xticks([]), plt.yticks([])

plt.tight_layout()
plt.subplots_adjust(wspace=0.1, hspace=0.1)
plt.show()

Sharpening은 5~6이 적합하다고 판단 됨. 
잘 안보이는 약들을 중심으로 Sharpening레벨을 테스트

022074
022347
019607
021771
016688

에 대해서 확인하고자 합니다.

# Sharpening레벨 테스트

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import random

# ==========================================
# 🔧 전처리 함수 (소수점 샤프닝 구현)
# ==========================================

def get_base_sharp_img(img, level):
    """정수 레벨의 샤프닝 이미지를 생성하는 함수"""
    if level == 4: # 매우 강함
        kernel = np.array([[-1, -2, -1], [-2, 13, -2], [-1, -2, -1]])
    elif level == 5: # 극한
        kernel = np.array([[-2, -2, -2], [-2, 17, -2], [-2, -2, -2]])
    elif level == 6: # 초극한 (5.25 구현용)
        kernel = np.array([[-2, -3, -2], [-3, 21, -3], [-2, -3, -2]])
    else:
        return img
    return cv2.filter2D(img, -1, kernel)

def apply_preprocessing(img_bgr, sharp_val):
    # 1. CLAHE 적용 (L 채널, Limit 2.0 고정)
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(img_lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_clahe = cv2.merge((l, a, b))
    img_clahe = cv2.cvtColor(img_clahe, cv2.COLOR_LAB2BGR)

    # 2. Bilateral Filter (Level 1 고정: d=5, sigma=50)
    img_bi = cv2.bilateralFilter(img_clahe, d=5, sigmaColor=50, sigmaSpace=50)

    # 3. Sharpening (소수점 블렌딩 로직)
    # 예: 4.25 -> Level 4 (75%) + Level 5 (25%)
    
    val_floor = int(sharp_val)      # 예: 4
    val_ceil = val_floor + 1        # 예: 5
    ratio = sharp_val - val_floor   # 예: 0.25

    img_floor = get_base_sharp_img(img_bi, val_floor)
    img_ceil = get_base_sharp_img(img_bi, val_ceil)

    # 두 이미지를 비율에 맞춰 합성 (Weighted Add)
    # alpha(floor) = 1 - ratio, beta(ceil) = ratio
    img_result = cv2.addWeighted(img_floor, 1 - ratio, img_ceil, ratio, 0)
    
    return cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB)


# ==========================================
# 📂 설정 및 데이터 로드
# ==========================================
# 🔥 NEW 폴더로 경로 변경
crop_root = r"C:\Users\home\Desktop\AI_study\processed_crops_NEW_ONLY1.2"

# 지정된 약물 ID 리스트 (5개)
target_drugs = ['022074', '022347', '019607', '021771', '016688']

# 샤프닝 테스트 레벨
sharp_levels = [4.0, 4.25, 4.5, 4.75, 5.0, 5.25]
col_titles = ["Original"] + [f"Sharp({v})" for v in sharp_levels]

# ==========================================
# 🖼️ 약물별 시각화 실행 (Loop)
# ==========================================

for drug_id in target_drugs:
    # 1. 해당 약물 폴더 찾기
    # 폴더명에 drug_id가 포함된 경로 검색
    found_folders = glob.glob(os.path.join(crop_root, f"*{drug_id}*"))
    
    if not found_folders:
        print(f"❌ 약물 폴더를 찾을 수 없음: {drug_id}")
        continue
    
    target_folder = found_folders[0] # 첫 번째 매칭 폴더 사용
    all_imgs = glob.glob(os.path.join(target_folder, "*.jpg"))
    
    # 2. 랜덤 10장 추출
    if len(all_imgs) >= 10:
        selected_imgs = random.sample(all_imgs, 10)
    else:
        selected_imgs = all_imgs # 10장 미만이면 전부 다
    
    print(f"💊 약물 [{drug_id}] 분석 시작... (샘플 {len(selected_imgs)}개)")

    # 3. 차트 그리기 (행: 10개, 열: 7개)
    plt.figure(figsize=(20, 30)) # 세로로 긴 차트
    
    for row, img_path in enumerate(selected_imgs):
        img_bgr = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        # (1) 원본
        images_to_show = [img_rgb]
        
        # (2) 샤프닝 단계별 생성
        for level in sharp_levels:
            processed = apply_preprocessing(img_bgr, sharp_val=level)
            images_to_show.append(processed)
        
        # (3) 한 줄 그리기
        for col, img in enumerate(images_to_show):
            idx = row * 7 + col + 1
            plt.subplot(len(selected_imgs), 7, idx)
            plt.imshow(img)
            
            # 맨 윗줄에만 컬럼 제목 표시
            if row == 0:
                plt.title(col_titles[col], fontsize=12, fontweight='bold', pad=10)
            
            # 맨 왼쪽 줄에만 파일명(일부) 표시
            if col == 0:
                fname = os.path.basename(img_path).split('_')[-1] # 파일명 뒤쪽만 표시
                plt.ylabel(f"Img {row+1}", fontsize=10, fontweight='bold')
            
            plt.xticks([]), plt.yticks([])
            
    plt.tight_layout()
    plt.suptitle(f"Drug ID: {drug_id} (CLAHE 2.0 / Bi 1 / Sharp Var)", fontsize=16, y=1.005)
    plt.show() # 약물 하나당 차트 하나씩 출력
    print(f"✅ [{drug_id}] 출력 완료.\n" + "="*50)

CLAHE 2.0 + Bi(1) 고정. Sharp(4, 4.25, 4.5, 4.75, 5, 5.25)에 대해 확인. 
위에 언급한대로, 022074, 022347, 019607, 021771, 016688에 대해서만 확인했습니다.
가장 구별이 안가는 조합들입니다. 

여러번 코드를 돌려 확인해봤습니다. 특히, 022074가 가장 어려운 약물 같습니다.
확인해보니, 4.5, 4.75, 5.0 외의 수치에서는 각인이 불분명한 경우가 있었습니다. 
따라서, 4.5, 4.75, 5.0 중 1개를 골라야하는데, 그다지 각인의 선명함에는 그렇게 큰 차이가 없습니다.
노이즈를 최소화하기 위해 4.5가 가장 최적의 수치라고 판단했습니다. 

# 전체 데이터 일괄 변환 코드

In [ ]:
import cv2
import numpy as np
import os
import glob
from tqdm import tqdm  # 진행률 표시바 (없으면 pip install tqdm)

# ==========================================
# 1. 최적화된 전처리 함수 (보고서 내용 적용)
# ==========================================
def apply_optimized_preprocessing(img_bgr):
    # CLAHE 2.0
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(img_lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_clahe = cv2.merge((l, a, b))
    img_clahe = cv2.cvtColor(img_clahe, cv2.COLOR_LAB2BGR)

    # Bilateral 1
    img_bi = cv2.bilateralFilter(img_clahe, d=5, sigmaColor=50, sigmaSpace=50)

    # Sharpening 4.5 (Blending)
    kernel_4 = np.array([[-1, -2, -1], [-2, 13, -2], [-1, -2, -1]])
    kernel_5 = np.array([[-2, -2, -2], [-2, 17, -2], [-2, -2, -2]])
    
    img_sharp_4 = cv2.filter2D(img_bi, -1, kernel_4)
    img_sharp_5 = cv2.filter2D(img_bi, -1, kernel_5)
    
    # 50:50 합성
    img_final = cv2.addWeighted(img_sharp_4, 0.5, img_sharp_5, 0.5, 0)
    return img_final

# ==========================================
# 2. 경로 설정
# ==========================================
# 원본 이미지가 있는 폴더 (1.2배 크롭된 원본들)
input_root = r"C:\Users\home\Desktop\AI_study\processed_crops_NEW_ONLY1.2"

# 전처리된 이미지를 저장할 새 폴더 이름
output_root = r"C:\Users\home\Desktop\AI_study\final_dataset_preprocessed"

# ==========================================
# 3. 일괄 변환 실행
# ==========================================
def process_all_images():
    # 모든 jpg 파일 찾기
    all_images = glob.glob(os.path.join(input_root, "**", "*.jpg"), recursive=True)
    print(f"📦 총 {len(all_images)}장의 이미지를 변환합니다...")

    os.makedirs(output_root, exist_ok=True)
    
    success_cnt = 0
    fail_cnt = 0

    # tqdm으로 진행 상황 보여주기
    for img_path in tqdm(all_images):
        try:
            # 이미지 로드
            img = cv2.imread(img_path)
            if img is None:
                fail_cnt += 1
                continue

            # 전처리 적용
            processed_img = apply_optimized_preprocessing(img)

            # 저장 경로 만들기 (폴더 구조 유지)
            # 예: input/K-001/a.jpg -> output/K-001/a.jpg
            relative_path = os.path.relpath(img_path, input_root)
            save_path = os.path.join(output_root, relative_path)
            
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            # 저장
            cv2.imwrite(save_path, processed_img)
            success_cnt += 1

        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            fail_cnt += 1

    print("-" * 50)
    print(f"✅ 변환 완료!")
    print(f"✨ 성공: {success_cnt}장")
    print(f"❌ 실패: {fail_cnt}장")
    print(f"📂 저장 위치: {output_root}")

if __name__ == "__main__":
    process_all_images()

# [실전용] Inference 파이프라인 예시

In [ ]:
# ==========================================
# 🚀 [실전용] Inference 파이프라인 예시
# ==========================================
import cv2
import torch

# 1. 모델 로드 (학습된 모델)
model = torch.load('best_model.pt') 

# 2. 새로운 이미지 도착 (원본)
raw_image = cv2.imread('new_pill_from_camera.jpg')

# ---------------------------------------------------------
# ⚠️ [핵심] 여기서 사용자님의 코드가 작동해야 합니다!
# ---------------------------------------------------------
processed_image = apply_optimized_preprocessing(raw_image) 
# (이 함수는 아까 만든 그 함수입니다: CLAHE+Bi+Sharp)
# ---------------------------------------------------------

# 3. 모델에게 전달 (전처리된 이미지를 줌)
# (YOLO나 PyTorch 모델이 요구하는 텐서 변환 등은 여기서 추가)
prediction = model(processed_image)

print(f"이 약은 {prediction} 입니다!")

In [ ]:
import os

# 1. 경로 설정 (사용자님의 실제 경로로 수정하세요)
img_dir = r'C:\Users\home\Desktop\AI_study\processed_crops_new_ONLY1.2'
ann_dir = r'C:\Users\home\Desktop\AI_study\processed_json_new_ONLY1.2'

# 2. 파일 목록 가져오기 (확장자 제외한 이름만 추출)
img_files = {os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))}
ann_files = {os.path.splitext(f)[0] for f in os.listdir(ann_dir) if f.endswith('.json')}

# 3. 차이 비교
only_in_img = img_files - ann_files
only_in_ann = ann_files - img_files

print(f"이미지 개수: {len(img_files)}")
print(f"JSON 개수: {len(ann_files)}")
print("-" * 30)

if only_in_img:
    print(f"⚠️ JSON이 없는 이미지 ({len(only_in_img)}개):")
    print(list(only_in_img)[:10], "... 등") # 너무 많을까봐 10개만 출력

if only_in_ann:
    print(f"⚠️ 이미지가 없는 JSON ({len(only_in_ann)}개):")
    print(list(only_in_ann)[:10], "... 등")

In [ ]:
from pathlib import Path

# ==========================================
# 1. 경로 설정 (사용자님의 실제 경로로 꼭 수정해주세요!)
# ==========================================
# 예: "C:/Users/name/project/data/train_images" 등 절대 경로로 적으면 더 확실합니다.
img_dir_path = r'C:\Users\home\Desktop\AI_study\processed_crops_new_ONLY1.2'
ann_dir_path = r'C:\Users\home\Desktop\AI_study\processed_json_new_ONLY1.2' 

# ==========================================
# 2. 하위 폴더까지 싹 다 검색 (Recursive Search)
# ==========================================
def get_file_map(root_dir, extensions):
    path_obj = Path(root_dir)
    file_map = {}
    
    # rglob('*')는 모든 하위 폴더를 다 뒤집니다
    for p in path_obj.rglob('*'):
        if p.is_file() and p.suffix.lower() in extensions:
            # 파일명(확장자 제외)을 키(Key)로, 전체 경로를 값(Value)으로 저장
            file_map[p.stem] = p
            
    return file_map

print("📂 파일 검색 중... (파일이 많으면 조금 걸릴 수 있습니다)")

# 이미지(.jpg, .png 등)와 JSON(.json) 찾기
img_files = get_file_map(img_dir_path, {'.jpg', '.jpeg', '.png', '.bmp'})
ann_files = get_file_map(ann_dir_path, {'.json'})

print(f"✅ 이미지 발견: {len(img_files)}개")
print(f"✅ JSON 발견 : {len(ann_files)}개")
print("-" * 40)

# ==========================================
# 3. 짝이 안 맞는 '범인' 색출
# ==========================================
img_set = set(img_files.keys())
ann_set = set(ann_files.keys())

# JSON이 없는 이미지 (이미지만 덩그러니 있는 경우)
only_in_img = img_set - ann_set
# 이미지가 없는 JSON (JSON만 덩그러니 있는 경우)
only_in_ann = ann_set - img_set

if not only_in_img and not only_in_ann:
    print("🎉 축하합니다! 모든 이미지와 JSON의 개수와 짝이 완벽하게 맞습니다.")
else:
    if only_in_img:
        print(f"⚠️ [문제 1] JSON이 없는 이미지: {len(only_in_img)}개")
        # 예시로 5개만 경로까지 출력
        for name in list(only_in_img)[:5]:
            print(f"   - {img_files[name]}")
        print("   ... (생략)")
        print("")

    if only_in_ann:
        print(f"⚠️ [문제 2] 이미지가 없는 JSON: {len(only_in_ann)}개")
        # 예시로 5개만 경로까지 출력
        for name in list(only_in_ann)[:5]:
            print(f"   - {ann_files[name]}")
        print("   ... (생략)")

In [ ]:
from pathlib import Path

# ==========================================
# 1. 경로 설정 (사용자님의 실제 경로로 꼭 수정해주세요!)
# ==========================================
# 예: "C:/Users/name/project/data/train_images" 등 절대 경로로 적으면 더 확실합니다.

img_dir_path = r'C:\Users\home\Desktop\AI_study\processed_crops_old_ONLY1.2'
ann_dir_path = r'C:\Users\home\Desktop\AI_study\processed_json_old_ONLY1.2' 

# ==========================================
# 2. 하위 폴더까지 싹 다 검색 (Recursive Search)
# ==========================================
def get_file_map(root_dir, extensions):
    path_obj = Path(root_dir)
    file_map = {}
    
    # rglob('*')는 모든 하위 폴더를 다 뒤집니다
    for p in path_obj.rglob('*'):
        if p.is_file() and p.suffix.lower() in extensions:
            # 파일명(확장자 제외)을 키(Key)로, 전체 경로를 값(Value)으로 저장
            file_map[p.stem] = p
            
    return file_map

print("📂 파일 검색 중... (파일이 많으면 조금 걸릴 수 있습니다)")

# 이미지(.jpg, .png 등)와 JSON(.json) 찾기
img_files = get_file_map(img_dir_path, {'.jpg', '.jpeg', '.png', '.bmp'})
ann_files = get_file_map(ann_dir_path, {'.json'})

print(f"✅ 이미지 발견: {len(img_files)}개")
print(f"✅ JSON 발견 : {len(ann_files)}개")
print("-" * 40)

# ==========================================
# 3. 짝이 안 맞는 '범인' 색출
# ==========================================
img_set = set(img_files.keys())
ann_set = set(ann_files.keys())

# JSON이 없는 이미지 (이미지만 덩그러니 있는 경우)
only_in_img = img_set - ann_set
# 이미지가 없는 JSON (JSON만 덩그러니 있는 경우)
only_in_ann = ann_set - img_set

if not only_in_img and not only_in_ann:
    print("🎉 축하합니다! 모든 이미지와 JSON의 개수와 짝이 완벽하게 맞습니다.")
else:
    if only_in_img:
        print(f"⚠️ [문제 1] JSON이 없는 이미지: {len(only_in_img)}개")
        # 예시로 5개만 경로까지 출력
        for name in list(only_in_img)[:5]:
            print(f"   - {img_files[name]}")
        print("   ... (생략)")
        print("")

    if only_in_ann:
        print(f"⚠️ [문제 2] 이미지가 없는 JSON: {len(only_in_ann)}개")
        # 예시로 5개만 경로까지 출력
        for name in list(only_in_ann)[:5]:
            print(f"   - {ann_files[name]}")
        print("   ... (생략)")

In [ ]:
import os
import shutil
from pathlib import Path

# ==========================================
# 1. 경로 설정
# ==========================================
# 사용자님의 경로 그대로 사용
img_dir_path = r'C:\Users\home\Desktop\AI_study\processed_crops_new_ONLY1.2'
ann_dir_path = r'C:\Users\home\Desktop\AI_study\processed_json_new_ONLY1.2'

# [수정] 쓰레기통을 AI_study 폴더 안에 만들어서 찾기 쉽게 변경했습니다.
trash_dir = r'C:\Users\home\Desktop\AI_study\_trash_images' 

# 폴더 없으면 생성
os.makedirs(trash_dir, exist_ok=True)

# ==========================================
# 2. 파일 짝 맞추기 (재검색)
# ==========================================
def get_file_map(root_dir, extensions):
    path_obj = Path(root_dir)
    file_map = {}
    # 하위 폴더까지 싹 다 뒤짐 (rglob)
    for p in path_obj.rglob('*'):
        if p.is_file() and p.suffix.lower() in extensions:
            file_map[p.stem] = p
    return file_map

print("🔍 다시 스캔 중...")
img_files = get_file_map(img_dir_path, {'.jpg', '.jpeg', '.png', '.bmp'})
ann_files = get_file_map(ann_dir_path, {'.json'})

# JSON이 없는 이미지 찾기
img_set = set(img_files.keys())
ann_set = set(ann_files.keys())
orphan_images = img_set - ann_set

print(f"⚠️ 발견된 짝 없는 이미지: {len(orphan_images)}개")

# ==========================================
# 3. 격리 조치 (Move)
# ==========================================
if orphan_images:
    # [수정] 변수명 오타 수정 (_trash_images -> trash_dir)
    print(f"\n🚚 {trash_dir} 폴더로 이동을 시작합니다...")
    
    count = 0
    for name in orphan_images:
        src_path = img_files[name]
        dst_path = os.path.join(trash_dir, src_path.name)
        
        try:
            shutil.move(str(src_path), dst_path)
            count += 1
        except Exception as e:
            print(f"❌ 이동 실패: {src_path.name} ({e})")

    print("-" * 30)
    print(f"✅ 총 {count}개의 이미지를 치웠습니다!")
    print(f"🗑️ 확인 후 '{trash_dir}' 폴더를 삭제하시면 됩니다.")
else:
    print("✨ 치울 파일이 없습니다. 완벽합니다!")

# 이미지 시각 후처리 코드V2

In [ ]:
import cv2
import numpy as np
import os
import glob
from tqdm import tqdm

# ==========================================
# 0. 한글 경로 지원 입출력 함수 (필수!)
# ==========================================
def imread_korean(path):
    with open(path, "rb") as f:
        bytes = bytearray(f.read())
        numpy_array = np.asarray(bytes, dtype=np.uint8)
        return cv2.imdecode(numpy_array, cv2.IMREAD_COLOR)

def imwrite_korean(path, img, params=None):
    try:
        ext = os.path.splitext(path)[1]
        result, n = cv2.imencode(ext, img, params)
        if result:
            with open(path, mode='w+b') as f:
                n.tofile(f)
            return True
        else:
            return False
    except Exception as e:
        print(f"Save Error: {e}")
        return False

# ==========================================
# 1. 최적화된 전처리 함수 (작성하신 그대로)
# ==========================================
def apply_optimized_preprocessing(img_bgr):
    # CLAHE 2.0
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(img_lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_clahe = cv2.merge((l, a, b))
    img_clahe = cv2.cvtColor(img_clahe, cv2.COLOR_LAB2BGR)

    # Bilateral 1
    img_bi = cv2.bilateralFilter(img_clahe, d=5, sigmaColor=50, sigmaSpace=50)

    # Sharpening 4.5 (Blending)
    kernel_4 = np.array([[-1, -2, -1], [-2, 13, -2], [-1, -2, -1]])
    kernel_5 = np.array([[-2, -2, -2], [-2, 17, -2], [-2, -2, -2]])
    
    img_sharp_4 = cv2.filter2D(img_bi, -1, kernel_4)
    img_sharp_5 = cv2.filter2D(img_bi, -1, kernel_5)
    
    # 50:50 합성
    img_final = cv2.addWeighted(img_sharp_4, 0.5, img_sharp_5, 0.5, 0)
    return img_final

# ==========================================
# 2. 경로 설정
# ==========================================
input_root = r"C:\Users\home\Desktop\AI_study\processed_crops_old_ONLY1.2"
output_root = r"C:\Users\home\Desktop\AI_study\final_dataset_preprocessed_old_ONLY1.2"

# ==========================================
# 3. 일괄 변환 실행
# ==========================================
def process_all_images():
    # 확장자 다양하게 찾기 (jpg, jpeg, png)
    extensions = ['*.jpg', '*.jpeg', '*.png']
    all_images = []
    
    for ext in extensions:
        all_images.extend(glob.glob(os.path.join(input_root, "**", ext), recursive=True))
        
    print(f"📦 총 {len(all_images)}장의 이미지를 변환합니다...")

    os.makedirs(output_root, exist_ok=True)
    
    success_cnt = 0
    fail_cnt = 0

    for img_path in tqdm(all_images):
        try:
            # [수정] 한글 경로 안전 로딩
            img = imread_korean(img_path)
            
            if img is None:
                fail_cnt += 1
                continue

            # 전처리 적용
            processed_img = apply_optimized_preprocessing(img)

            # 저장 경로 만들기
            relative_path = os.path.relpath(img_path, input_root)
            save_path = os.path.join(output_root, relative_path)
            
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            # [수정] 한글 경로 안전 저장
            if imwrite_korean(save_path, processed_img):
                success_cnt += 1
            else:
                fail_cnt += 1

        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            fail_cnt += 1

    print("-" * 50)
    print(f"✅ 변환 완료!")
    print(f"✨ 성공: {success_cnt}장")
    print(f"❌ 실패: {fail_cnt}장")
    print(f"📂 저장 위치: {output_root}")

if __name__ == "__main__":
    process_all_images()